# ChemAI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chembl_webresource_client.new_client import new_client

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
pd.set_option('display.max_columns', 120)

## 1. Kaggle data и baseline

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

In [ ]:
target_columns = ['IC50, mM', 'CC50, mM', 'SI']

X = train.drop(columns=target_columns)
y = train[target_columns]

imputer = SimpleImputer(strategy='median')

X_imputed = imputer.fit_transform(X)
test_imputed = imputer.transform(test)

X = pd.DataFrame(X_imputed, columns=X.columns)
test = pd.DataFrame(test_imputed, columns=test.columns)

In [ ]:
target_columns = ['IC50, mM', 'CC50, mM', 'SI']
submission_columns = ['IC50', 'CC50', 'SI']

feature_columns = [
    col for col in train.columns
    if col not in target_columns + ['index']
]

X_raw = X[feature_columns].copy()
X_test_raw = test[feature_columns].copy()
y_raw = y[target_columns].copy()

print('Train:', train.shape)
print('Test:', test.shape)
print('Features:', len(feature_columns))

In [ ]:
limits = {
    'IC50, mM': (0, 20000),
    'CC50, mM': (0, 20000),
    'SI':       (0, 250),
}

y_clipped = y_raw.copy()
y_clipped[list(limits)] = y_clipped[list(limits)].clip(
    lower={k: v[0] for k, v in limits.items()},
    upper={k: v[1] for k, v in limits.items()}
)

imputer_full = SimpleImputer(strategy='median')

X = pd.DataFrame(
    imputer_full.fit_transform(X_raw),
    columns=feature_columns
)

X_test = pd.DataFrame(
    imputer_full.transform(X_test_raw),
    columns=feature_columns
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y_clipped,
    test_size=0.2,
    random_state=RANDOM_STATE
)

baseline_model = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

baseline_model.fit(X_train, y_train)
valid_pred = baseline_model.predict(X_valid)

print('Baseline validation scores:')
for i, target in enumerate(target_columns):
    mae = mean_absolute_error(y_valid.iloc[:, i], valid_pred[:, i])
    rmse = mean_squared_error(y_valid.iloc[:, i], valid_pred[:, i]) ** 0.5
    r2 = r2_score(y_valid.iloc[:, i], valid_pred[:, i])
    print(f'{target}: MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}')

baseline_model.fit(X, y_clipped)
baseline_pred = baseline_model.predict(X_test)

baseline_submission = sample_submission.copy()
baseline_submission[submission_columns] = baseline_pred


baseline_submission.to_csv('submission_baseline_clipped.csv', index=False)
print('Saved submission_baseline_clipped.csv')

## 2. Загрузка ChEMBL
target_ids взяты с https://www.ebi.ac.uk/ по фильтру "organism=Influenza"

In [ ]:
target = new_client.target
activity = new_client.activity

target_ids = [
    'CHEMBL355',
    'CHEMBL613740',
    'CHEMBL613128',
    'CHEMBL612783',
    'CHEMBL613129',
    'CHEMBL2366902',
    'CHEMBL2367089',
    'CHEMBL612610',
]

activity_fields = [
    'activity_id',
    'molecule_chembl_id',
    'canonical_smiles',
    'standard_type',
    'standard_relation',
    'standard_value',
    'standard_units',
    'pchembl_value',
    'assay_chembl_id',
    'target_chembl_id',
    'document_chembl_id'
]

In [ ]:
# 1. Загружаем информацию по target
target_info_rows = []

for target_id in target_ids:
    try:
        info = target.get(target_id)
        target_info_rows.append({
            'target_chembl_id': info.get('target_chembl_id'),
            'pref_name': info.get('pref_name'),
            'organism': info.get('organism'),
            'target_type': info.get('target_type'),
            'tax_id': info.get('tax_id'),
        })
    except Exception as e:
        print('Target error:', target_id, e)

targets_df = pd.DataFrame(target_info_rows)

print('Targets:')
display(targets_df)

In [ ]:
# 2. Загружаем активности по всем target
activity_rows = []

for i, target_id in enumerate(target_ids):
    print(f'Loading {i + 1}/{len(target_ids)}: {target_id}')
    
    try:
        acts = activity.filter(
            target_chembl_id=target_id
        ).only(activity_fields)
        
        temp = pd.DataFrame(list(acts))
        
        if len(temp) > 0:
            temp['source_target_chembl_id'] = target_id
            activity_rows.append(temp)
            print('  rows:', len(temp))
        else:
            print('  no activities')
            
    except Exception as e:
        print('  Activity error:', target_id, e)

acts_df = pd.concat(activity_rows, ignore_index=True)

print('All activities shape:', acts_df.shape)
display(acts_df.head())

In [ ]:
# 3. Добавляем organism / target name к активностям
acts_df = acts_df.merge(
    targets_df,
    on='target_chembl_id',
    how='left'
)

# Удаляем дубли, если одна и та же activity попала через разные target
acts_df = acts_df.drop_duplicates()

print(acts_df.shape)
display(acts_df.head())

display(acts_df['organism'].value_counts(dropna=False))
display(acts_df['standard_type'].value_counts().head(40))
display(acts_df['standard_units'].value_counts(dropna=False))

In [ ]:
ic50_df = acts_df[
    acts_df['standard_type'] == 'IC50'
].copy()

ic50_df['standard_value'] = pd.to_numeric(
    ic50_df['standard_value'],
    errors='coerce'
)

ic50_df = ic50_df.dropna(subset=['standard_value'])

# Оставляем точные значения
ic50_df = ic50_df[
    ic50_df['standard_relation'].isna() |
    (ic50_df['standard_relation'] == '=')
].copy()

# Оставляем нормальные концентрационные единицы
ic50_df = ic50_df[
    ic50_df['standard_units'].isin(['nM', 'uM', 'µM', 'mM'])
].copy()

print(ic50_df.shape)
display(ic50_df.head())
display(ic50_df['standard_units'].value_counts())
display(ic50_df['organism'].value_counts())

In [ ]:
acts_df.to_csv('chembl_multiple_influenza_targets_activities.csv', index=False)
ic50_df.to_csv('chembl_multiple_influenza_targets_ic50.csv', index=False)

print('Saved files')

## 3. Оставляем IC50 и добавляем assay descriptions

In [ ]:
chembl_ic50_long = acts_df[
    acts_df['standard_type'].eq('IC50')
].copy()

chembl_ic50_long['standard_value'] = pd.to_numeric(
    chembl_ic50_long['standard_value'],
    errors='coerce'
)

chembl_ic50_long = chembl_ic50_long.dropna(
    subset=['standard_value', 'canonical_smiles', 'assay_chembl_id']
)

print('IC50 rows:', chembl_ic50_long.shape)
display(chembl_ic50_long['standard_units'].value_counts())
display(chembl_ic50_long.head())

In [ ]:
assay = new_client.assay

assay_ids = chembl_ic50_long['assay_chembl_id'].dropna().unique().tolist()
assay_rows = []

for assay_id in assay_ids:
    try:
        item = assay.get(assay_id)
        assay_rows.append({
            'assay_chembl_id': assay_id,
            'description': item.get('description'),
            'assay_type': item.get('assay_type'),
            'target_chembl_id_assay': item.get('target_chembl_id'),
            'document_chembl_id_assay': item.get('document_chembl_id')
        })
    except Exception as e:
        print('Error:', assay_id, e)

assays_df = pd.DataFrame(assay_rows)

chembl_ic50_long = chembl_ic50_long.merge(
    assays_df,
    on='assay_chembl_id',
    how='left'
)

print(chembl_ic50_long.shape)
display(chembl_ic50_long.head())

## 4. Единицы измерения

In [ ]:
def convert_ic50_units(value, units):
    if pd.isna(value):
        return pd.Series({'IC50_nM': np.nan, 'IC50_uM': np.nan, 'IC50_mM': np.nan})

    if units == 'nM':
        return pd.Series({
            'IC50_nM': value,
            'IC50_uM': value / 1000,
            'IC50_mM': value / 1_000_000
        })

    if units in ['uM', 'µM', 'umol/L']:
        return pd.Series({
            'IC50_nM': value * 1000,
            'IC50_uM': value,
            'IC50_mM': value / 1000
        })

    if units == 'mM':
        return pd.Series({
            'IC50_nM': value * 1_000_000,
            'IC50_uM': value * 1000,
            'IC50_mM': value
        })

    return pd.Series({'IC50_nM': np.nan, 'IC50_uM': np.nan, 'IC50_mM': np.nan})

unit_values = chembl_ic50_long.apply(
    lambda row: convert_ic50_units(row['standard_value'], row['standard_units']),
    axis=1
)

chembl_ic50_long = pd.concat(
    [chembl_ic50_long.reset_index(drop=True), unit_values.reset_index(drop=True)],
    axis=1
)

chembl_ic50_long = chembl_ic50_long[
    chembl_ic50_long['IC50_uM'].notna() &
    (chembl_ic50_long['IC50_uM'] > 0)
].copy()

print('Rows with molar IC50:', chembl_ic50_long.shape)
display(chembl_ic50_long[['standard_value', 'standard_units', 'IC50_nM', 'IC50_uM', 'IC50_mM']].head())
display(chembl_ic50_long['standard_units'].value_counts())

In [ ]:
display(chembl_ic50_long[['standard_value', 'standard_units', 'IC50_nM', 'IC50_uM', 'IC50_mM']].describe())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Kaggle IC50
kaggle_ic50 = train['IC50, mM'].dropna()
kaggle_ic50 = kaggle_ic50[kaggle_ic50 > 0]

chembl_ic50_units = chembl_ic50_long[
    (chembl_ic50_long['standard_type'] == 'IC50')
    & (chembl_ic50_long['standard_value'].notna())
    & (chembl_ic50_long['standard_value'] > 0)
    & (chembl_ic50_long['standard_value'] < 5000000)  # <------------------ тут можно фильтровать chembl по верхнему порогу
].copy()

# Переводим ChEMBL в разные шкалы
chembl_ic50_units['IC50_nM'] = np.nan
chembl_ic50_units['IC50_uM'] = np.nan
chembl_ic50_units['IC50_mM'] = np.nan

mask_nm = chembl_ic50_units['standard_units'] == 'nM'
mask_um = chembl_ic50_units['standard_units'].isin(['uM', 'µM', 'umol/L'])
mask_mm = chembl_ic50_units['standard_units'] == 'mM'

# Если исходно nM
chembl_ic50_units.loc[mask_nm, 'IC50_nM'] = chembl_ic50_units.loc[mask_nm, 'standard_value']
chembl_ic50_units.loc[mask_nm, 'IC50_uM'] = chembl_ic50_units.loc[mask_nm, 'standard_value'] / 1000
chembl_ic50_units.loc[mask_nm, 'IC50_mM'] = chembl_ic50_units.loc[mask_nm, 'standard_value'] / 1_000_000

# Если исходно uM
chembl_ic50_units.loc[mask_um, 'IC50_nM'] = chembl_ic50_units.loc[mask_um, 'standard_value'] * 1000
chembl_ic50_units.loc[mask_um, 'IC50_uM'] = chembl_ic50_units.loc[mask_um, 'standard_value']
chembl_ic50_units.loc[mask_um, 'IC50_mM'] = chembl_ic50_units.loc[mask_um, 'standard_value'] / 1000

# Если исходно mM
chembl_ic50_units.loc[mask_mm, 'IC50_nM'] = chembl_ic50_units.loc[mask_mm, 'standard_value'] * 1_000_000
chembl_ic50_units.loc[mask_mm, 'IC50_uM'] = chembl_ic50_units.loc[mask_mm, 'standard_value'] * 1000
chembl_ic50_units.loc[mask_mm, 'IC50_mM'] = chembl_ic50_units.loc[mask_mm, 'standard_value']

# Убираем пустые значения
chembl_nM = chembl_ic50_units['IC50_nM'].dropna()
chembl_uM = chembl_ic50_units['IC50_uM'].dropna()
chembl_mM = chembl_ic50_units['IC50_mM'].dropna()

chembl_nM = chembl_nM[chembl_nM > 0]
chembl_uM = chembl_uM[chembl_uM > 0]
chembl_mM = chembl_mM[chembl_mM > 0]

print('Counts:')
print('Kaggle:', len(kaggle_ic50))
print('ChEMBL nM:', len(chembl_nM))
print('ChEMBL uM:', len(chembl_uM))
print('ChEMBL mM:', len(chembl_mM))

In [ ]:
summary = pd.DataFrame({
    'Kaggle_IC50_column': kaggle_ic50.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    ),
    'ChEMBL_IC50_nM': chembl_nM.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    ),
    'ChEMBL_IC50_uM': chembl_uM.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    ),
    'ChEMBL_IC50_mM': chembl_mM.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    ),
})

display(summary)

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(kaggle_ic50, bins=50, alpha=0.5, label='Kaggle IC50 column')
plt.hist(chembl_nM, bins=50, alpha=0.5, label='ChEMBL IC50 as nM')
plt.hist(chembl_uM, bins=50, alpha=0.5, label='ChEMBL IC50 as uM')
plt.hist(chembl_mM, bins=50, alpha=0.5, label='ChEMBL IC50 as mM')
plt.xlabel('IC50 value')
plt.ylabel('Count')
plt.title('IC50 distribution: original scale')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(np.log10(kaggle_ic50), bins=50, alpha=0.5, label='Kaggle IC50 column')
plt.hist(np.log10(chembl_nM), bins=50, alpha=0.5, label='ChEMBL IC50 as nM')
plt.hist(np.log10(chembl_uM), bins=50, alpha=0.5, label='ChEMBL IC50 as uM')
plt.hist(np.log10(chembl_mM), bins=50, alpha=0.5, label='ChEMBL IC50 as mM')
plt.xlabel('log10(IC50 value)')
plt.ylabel('Count')
plt.title('IC50 distribution: log10 scale')
plt.legend()
plt.show()

In [ ]:
plot_df = pd.DataFrame({
    'Kaggle': np.log10(kaggle_ic50),
    'ChEMBL_nM': pd.Series(np.log10(chembl_nM)).reset_index(drop=True),
    'ChEMBL_uM': pd.Series(np.log10(chembl_uM)).reset_index(drop=True),
    'ChEMBL_mM': pd.Series(np.log10(chembl_mM)).reset_index(drop=True),
})

plt.figure(figsize=(10, 5))
plt.boxplot(
    [plot_df[col].dropna() for col in plot_df.columns],
    labels=plot_df.columns,
    vert=True
)
plt.ylabel('log10(IC50 value)')
plt.title('IC50 scale comparison: log10 boxplot')
plt.grid(True)
plt.show()

In [ ]:
def plot_ecdf(values, label):
    values = np.sort(np.asarray(values))
    y = np.arange(1, len(values) + 1) / len(values)
    plt.plot(values, y, label=label)

plt.figure(figsize=(10, 5))
plot_ecdf(np.log10(kaggle_ic50), 'Kaggle')
plot_ecdf(np.log10(chembl_nM), 'ChEMBL as nM')
plot_ecdf(np.log10(chembl_uM), 'ChEMBL as uM')
plot_ecdf(np.log10(chembl_mM), 'ChEMBL as mM')
plt.xlabel('log10(IC50 value)')
plt.ylabel('Cumulative share')
plt.title('IC50 distribution comparison: ECDF')
plt.legend()
plt.grid(True)
plt.show()

## 5. RDKit-дескрипторы для ChEMBL

Считаем только те RDKit-дескрипторы, имена которых совпадают с признаками Kaggle.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator

rdkit_descriptor_names = [name for name, func in Descriptors._descList]

common_descriptor_names = [
    col for col in feature_columns
    if col in rdkit_descriptor_names
]

missing_from_rdkit = [
    col for col in feature_columns
    if col not in rdkit_descriptor_names
]

print('Kaggle features:', len(feature_columns))
print('Common RDKit descriptors:', len(common_descriptor_names))
print('Missing from RDKit calculator:', len(missing_from_rdkit))
print(missing_from_rdkit[:40])

In [ ]:
# Считаем дескрипторы для уникальных SMILES, чтобы не пересчитывать дубли
unique_molecules = (
    chembl_ic50_long[['molecule_chembl_id', 'canonical_smiles']]
    .drop_duplicates()
    .reset_index(drop=True)
)

calculator = MolecularDescriptorCalculator(common_descriptor_names)

def calc_common_rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return pd.Series(
            [np.nan] * len(common_descriptor_names),
            index=common_descriptor_names
        )

    try:
        values = calculator.CalcDescriptors(mol)
    except Exception:
        values = [np.nan] * len(common_descriptor_names)

    return pd.Series(values, index=common_descriptor_names)

chembl_desc = unique_molecules['canonical_smiles'].apply(calc_common_rdkit_descriptors)

chembl_desc = chembl_desc.replace([np.inf, -np.inf], np.nan)

chembl_desc_long = pd.concat(
    [unique_molecules.reset_index(drop=True), chembl_desc.reset_index(drop=True)],
    axis=1
)

print(chembl_desc_long.shape)
display(chembl_desc_long.head())

## 7. Molecule-level ChEMBL dataset для обучения

Для обучения делаем агрегирование по молекуле.  
Медиана нужна, чтобы молекулы с большим числом измерений не получили чрезмерный вес.

Если не агрегировать, одна молекула с 20 assays попадёт в train 20 раз и будет искусственно переважена.

In [ ]:
# Выбор шкалы для обучения ChEMBL IC50.
CHEMBL_IC50_TARGET = 'IC50_uM_median'

chembl_mol = (
    chembl_ic50_long[(chembl_ic50_long['standard_value'] < 4100000)]
    .groupby(['molecule_chembl_id', 'canonical_smiles'], as_index=False)
    .agg(
        IC50_nM_median=('IC50_nM', 'median'),
        IC50_uM_median=('IC50_uM', 'median'),
        IC50_mM_median=('IC50_mM', 'median'),
        IC50_uM_mean=('IC50_uM', 'mean'),
        IC50_uM_std=('IC50_uM', 'std'),
        IC50_uM_min=('IC50_uM', 'min'),
        IC50_uM_max=('IC50_uM', 'max'),
        IC50_n_measurements=('IC50_uM', 'size'),
        n_assays=('assay_chembl_id', 'nunique')
    )
)

chembl_mol['IC50_uM_std'] = chembl_mol['IC50_uM_std'].fillna(0)

chembl_aug = chembl_mol.merge(
    chembl_desc_long,
    on=['molecule_chembl_id', 'canonical_smiles'],
    how='left'
)

print(chembl_aug.shape)
display(chembl_aug.head())
display(chembl_aug[CHEMBL_IC50_TARGET].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

## 9. Подготовка матриц для IC50 augmentation

In [ ]:
def clean_feature_matrix(df, max_abs_value=1e12):
    df = df.copy()
    df = df.apply(pd.to_numeric, errors='coerce')
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.mask(df.abs() > max_abs_value, np.nan)
    return df

def impute_and_clean(X_train_like, X_external, X_test_like, columns, max_abs_value=1e12):
    X_train_like = clean_feature_matrix(X_train_like[columns], max_abs_value=max_abs_value)
    X_external = clean_feature_matrix(X_external[columns], max_abs_value=max_abs_value)
    X_test_like = clean_feature_matrix(X_test_like[columns], max_abs_value=max_abs_value)

    imputer = SimpleImputer(strategy='median')

    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train_like),
        columns=columns,
        index=X_train_like.index
    )

    X_external_imp = pd.DataFrame(
        imputer.transform(X_external),
        columns=columns,
        index=X_external.index
    )

    X_test_imp = pd.DataFrame(
        imputer.transform(X_test_like),
        columns=columns,
        index=X_test_like.index
    )

    # Финальная защита после imputation
    fill_values = X_train_imp.median()

    for df in [X_train_imp, X_external_imp, X_test_imp]:
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.mask(df.abs() > max_abs_value, np.nan, inplace=True)
        df.fillna(fill_values, inplace=True)

    return X_train_imp, X_external_imp, X_test_imp

In [ ]:
X_kaggle_ic50 = train[common_descriptor_names].copy()
y_kaggle_ic50 = y_raw['IC50, mM'].copy()

X_test_common = test[common_descriptor_names].copy()

X_chembl_ic50 = chembl_aug[common_descriptor_names].copy()
y_chembl_ic50 = chembl_aug[CHEMBL_IC50_TARGET].copy()

valid_chembl_mask = (
    y_chembl_ic50.notna() &
    (y_chembl_ic50 > 0) &
    (X_chembl_ic50.isna().mean(axis=1) < 0.2)
)

X_chembl_ic50 = X_chembl_ic50[valid_chembl_mask].copy()
y_chembl_ic50 = y_chembl_ic50[valid_chembl_mask].copy()

X_kaggle_ic50_imp, X_chembl_ic50_imp, X_test_common_imp = impute_and_clean(
    X_kaggle_ic50,
    X_chembl_ic50,
    X_test_common,
    common_descriptor_names,
    max_abs_value=1e12
)

print('Kaggle IC50 train:', X_kaggle_ic50_imp.shape)
print('ChEMBL IC50 external:', X_chembl_ic50_imp.shape)
print('Test common:', X_test_common_imp.shape)

print('Kaggle finite:', np.isfinite(X_kaggle_ic50_imp.to_numpy()).all())
print('ChEMBL finite:', np.isfinite(X_chembl_ic50_imp.to_numpy()).all())
print('Test finite:', np.isfinite(X_test_common_imp.to_numpy()).all())

print('Max abs Kaggle:', np.nanmax(np.abs(X_kaggle_ic50_imp.to_numpy())))
print('Max abs ChEMBL:', np.nanmax(np.abs(X_chembl_ic50_imp.to_numpy())))
print('Max abs Test:', np.nanmax(np.abs(X_test_common_imp.to_numpy())))

## 10. Holdout sanity check: Kaggle-only vs Kaggle+ChEMBL

In [ ]:
X_train_c, X_valid_c, y_train_c, y_valid_c = train_test_split(
    X_kaggle_ic50_imp,
    y_kaggle_ic50,
    test_size=0.2,
    random_state=RANDOM_STATE
)

CHEMBL_WEIGHT = 1

ic50_base_common = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

ic50_base_common.fit(X_train_c, y_train_c)
ic50_base_valid_pred = ic50_base_common.predict(X_valid_c)

X_aug_train = pd.concat(
    [X_train_c.reset_index(drop=True), X_chembl_ic50_imp.reset_index(drop=True)],
    axis=0,
    ignore_index=True
)

y_aug_train = pd.concat(
    [y_train_c.reset_index(drop=True), y_chembl_ic50.reset_index(drop=True)],
    axis=0,
    ignore_index=True
)

sample_weight_aug = np.concatenate([
    np.ones(len(X_train_c)),
    np.ones(len(X_chembl_ic50_imp)) * CHEMBL_WEIGHT
])

ic50_aug_common = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

ic50_aug_common.fit(
    X_aug_train,
    y_aug_train,
    sample_weight=sample_weight_aug
)

ic50_aug_valid_pred = ic50_aug_common.predict(X_valid_c)

print('IC50 baseline common descriptors:')
print('MAE:', mean_absolute_error(y_valid_c, ic50_base_valid_pred))
print('RMSE:', mean_squared_error(y_valid_c, ic50_base_valid_pred) ** 0.5)

print('\nIC50 augmented with ChEMBL:')
print('MAE:', mean_absolute_error(y_valid_c, ic50_aug_valid_pred))
print('RMSE:', mean_squared_error(y_valid_c, ic50_aug_valid_pred) ** 0.5)

## 11. Финальная IC50-модель и submissions

In [ ]:
X_aug_full = pd.concat(
    [X_kaggle_ic50_imp.reset_index(drop=True), X_chembl_ic50_imp.reset_index(drop=True)],
    axis=0,
    ignore_index=True
)

y_aug_full = pd.concat(
    [y_kaggle_ic50.reset_index(drop=True), y_chembl_ic50.reset_index(drop=True)],
    axis=0,
    ignore_index=True
)

sample_weight_full = np.concatenate([
    np.ones(len(X_kaggle_ic50_imp)),
    np.ones(len(X_chembl_ic50_imp)) * CHEMBL_WEIGHT
])

print('X_aug_full shape:', X_aug_full.shape)
print('y_aug_full shape:', y_aug_full.shape)
print('sample_weight_full shape:', sample_weight_full.shape)

print('X_aug_full finite:', np.isfinite(X_aug_full.to_numpy()).all())
print('y_aug_full finite:', np.isfinite(y_aug_full.to_numpy()).all())
print('sample_weight_full finite:', np.isfinite(sample_weight_full).all())
print('Max abs X_aug_full:', np.nanmax(np.abs(X_aug_full.to_numpy())))

ic50_aug_final = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

ic50_aug_final.fit(
    X_aug_full,
    y_aug_full,
    sample_weight=sample_weight_full
)

ic50_aug_test_pred = ic50_aug_final.predict(X_test_common_imp)
ic50_aug_test_pred = np.clip(ic50_aug_test_pred, 0, None)

print(pd.Series(ic50_aug_test_pred).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
# Полная замена IC50 на augmented prediction
submission_aug_ic50 = baseline_submission.copy()
submission_aug_ic50['IC50'] = ic50_aug_test_pred
submission_aug_ic50['IC50'] = submission_aug_ic50['IC50'].clip(lower=0)

submission_aug_ic50.to_csv('submission_augmented_ic50_only.csv', index=False)
print('Saved submission_augmented_ic50_only.csv')

# Blend: alpha — доля baseline IC50
for alpha in [0.25, 0.5, 0.75, 0.9]:
    sub = baseline_submission.copy()

    sub['IC50'] = (
        alpha * baseline_submission['IC50'].values +
        (1 - alpha) * ic50_aug_test_pred
    )

    sub['IC50'] = sub['IC50'].clip(lower=0)

    filename = f'submission_augmented_ic50_blend_alpha_{alpha}.csv'
    sub.to_csv(filename, index=False)
    print('Saved', filename)